# Demo 1: Phân tích Cú pháp & Phân vùng Báo cáo tài chính SEC 10-K (HTML Parsing & Section Detection)

## 1. Lý thuyết & Khái niệm
Báo cáo SEC 10-K là báo cáo tài chính thường niên bắt buộc của các công ty đại chúng tại Mỹ. Một báo cáo thường dài hàng trăm trang và chứa rất nhiều thông tin nhiễu. Để RAG hoạt động hiệu quả, chúng ta phải thực hiện giai đoạn **Document Ingestion (Tiền xử lý)** gồm hai bước nhỏ:
1. **Parsing (Phân tích cú pháp):** Bóc tách văn bản thô (clean text) từ mã HTML/XBRL phức tạp và loại bỏ các bảng biểu trống, thẻ CSS hoặc mã Javascript rác.
2. **Section Detection (Phân vùng báo cáo):** Sử dụng các biểu thức chính quy (Regex) làm bộ định tuyến để chia nhỏ báo cáo thành các mục (Items) cốt lõi như:
   * **Item 1:** Business (Mô tả kinh doanh)
   * **Item 1A:** Risk Factors (Các yếu tố rủi ro - Rất quan trọng để đánh giá rủi ro)
   * **Item 7:** MD&A (Thảo luận và phân tích của ban giám đốc - Chứa giải trình số liệu)
   * **Item 8:** Financial Statements (Bảng báo cáo tài chính - Chứa các bảng số liệu cân đối kế toán, dòng tiền)

### Quy trình Dữ liệu (Data Flow):
* **Input:** File HTML thô (.html) tải trực tiếp từ cơ sở dữ liệu SEC EDGAR.
* **Output:** Các đoạn văn bản sạch (clean text) phân biệt rõ ràng theo từng mục (Item) kèm theo metadata tương ứng.
* **Mục đích của Output:** Các đoạn văn bản sạch này sẽ được chuyển trực tiếp sang giai đoạn tiếp theo là **Chunking (Phân mảnh)**. Chúng không được đưa thẳng vào LLM vì kích thước một Item vẫn quá lớn so với giới hạn ngữ cảnh (Context Window) của mô hình và gây loãng thông tin.

## 2. Khởi tạo mã nguồn HTML giả lập
Chúng ta sẽ dựng một đoạn mã HTML thu nhỏ mô phỏng cấu trúc thực tế của một file báo cáo 10-K.

In [ ]:
html_content = """
<html>
<head><style>.header { color: blue; }</style></head>
<body>
    <div class="header">REGISTRATION STATEMENT UNDER THE SECURITIES ACT OF 1933</div>
    
    <!-- MÔ PHỎNG ITEM 1A -->
    <h2 style="text-align:center;">Item 1A. Risk Factors</h2>
    <p>Our business is subject to significant risks. First, global supply chain constraints and component shortages, especially in semiconductors, could materially affect our shipments.</p>
    <p>Second, intense competition in the smartphone and cloud service markets could reduce our gross margin.</p>
    
    <!-- MÔ PHỎNG ITEM 7 -->
    <h2 style="text-align:center;">Item 7. Management's Discussion and Analysis of Financial Condition</h2>
    <p>During fiscal year 2023, our net sales increased by 8% to $383,285 million compared to fiscal year 2022.</p>
    <p>Operating income was $114,301 million, representing a strong performance driven by services growth.</p>
    
    <!-- MÔ PHỎNG ITEM 8 -->
    <h2>Item 8. Financial Statements and Supplementary Data</h2>
    <table border="1">
        <tr><th>Year Ended</th><th>Net Income</th></tr>
        <tr><td>September 30, 2023</td><td>$96,995 million</td></tr>
        <tr><td>September 24, 2022</td><td>$99,803 million</td></tr>
    </table>
</body>
</html>
"""
print("=== INPUT (Mã HTML Thô) ===")
print(html_content[:300] + "\n...[truncated]...")

## 3. Parsing: Bóc tách text bằng BeautifulSoup4
Loại bỏ toàn bộ các thẻ HTML, CSS, giữ lại cấu trúc văn bản thô.

In [ ]:
from bs4 import BeautifulSoup
import re

print("[Log] Đang khởi chạy BeautifulSoup4 để bóc tách HTML...")
soup = BeautifulSoup(html_content, "html.parser")

# Loại bỏ các thẻ không chứa nội dung đọc được
for tag in soup(["style", "script", "head", "title", "meta"]):
    tag.decompose()

clean_text = soup.get_text(separator="\n")
print("[SUCCESS] Đã chuyển đổi sang văn bản sạch thành công!")
print("\n=== OUTPUT (Văn bản thô sau khi parse) ===")
print(clean_text)

## 4. Section Detection: Phân vùng văn bản bằng Regex
Chúng ta thiết kế biểu thức chính quy (Regex) để phát hiện ranh giới của từng Item và tách chúng ra thành các đoạn văn bản độc lập.

In [ ]:
# Định nghĩa các ranh giới Regex cho các Item mục tiêu
patterns = {
    "Item 1A": r"item\s*1a[.\s]*risk\s*factors",
    "Item 7": r"item\s*7[.\s]*management's\s*discussion",
    "Item 8": r"item\s*8[.\s]*financial\s*statements"
}

print("[Log] Bắt đầu quét tìm ranh giới các mục...")
positions = []
for item_name, pat in patterns.items():
    match = re.search(pat, clean_text, re.IGNORECASE)
    if match:
        start_pos = match.start()
        positions.append((item_name, start_pos))
        print(f"  - Phát hiện '{item_name}' tại ký tự thứ: {start_pos}")

# Sắp xếp các vị trí phát hiện theo thứ tự xuất hiện trong file
positions.sort(key=lambda x: x[1])

# Tiến hành cắt chuỗi văn bản theo các vị trí đã xác định
sections = {}
for i in range(len(positions)):
    item_name, start = positions[i]
    # Điểm kết thúc là điểm bắt đầu của Item kế tiếp, hoặc hết file
    end = positions[i+1][1] if i + 1 < len(positions) else len(clean_text)
    
    # Cắt chuỗi và chuẩn hóa khoảng trắng
    section_text = clean_text[start:end].strip()
    sections[item_name] = section_text
    print(f"[Log] Trích xuất thành công '{item_name}' (Độ dài: {len(section_text)} ký tự)")

print("\n" + "="*40)
print("KẾT QUẢ PHÂN VÙNG DỮ LIỆU ĐẦU RA")
print("="*40)
for item_name, text in sections.items():
    print(f"\n>>> {item_name} <<<")
    print(text[:300] + "...")

## 5. Kết luận và Chuyển giao dữ liệu
Đầu ra của giai đoạn này là các chuỗi văn bản sạch đã phân chia theo từng Item mục tiêu. 
Các chuỗi văn bản sạch này cùng với các thông tin nhãn (Metadata) như: `ticker = 'AAPL'`, `year = 2023`, `section = 'Item 7'` sẽ được chuyển trực tiếp sang làm đầu vào cho **Bước 2: Chunking (Phân mảnh)** để chia cắt thành các mảnh có kích thước đồng đều và chồng lấp ngữ cảnh.